# Baseline e Modelação

> Objetivo: construir e avaliar três modelos baseline em ordem: Regressão Logística, Árvore de Decisão e Random Forest.

## Estrutura desta secção
1. Preparação dos dados
2. Baseline: Regressão Logística
3. Baseline: Árvore de Decisão
4. Baseline: Random Forest

In [1]:
# importações de bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import preprocessing as prep

# Carregamento dos dados já com as transformações determinísticas de domínio
# aplicadas (999->NaN, recodificação do Sexo, fórmula do IMC, reconstrução do
# WAtotal_0, remoção de casos sem Grupo_pre e das variáveis pós-operatórias).
# A imputação e o one-hot encoding NÃO são feitos aqui: vivem dentro da pipeline,
# ajustada apenas com o treino (feedback #1 e #5). Por isso NÃO usamos o
# ortho_eda_clean.csv, que já traz a imputação feita sobre todas as linhas.
X, y = prep.carregar_dados()

In [2]:
# primeiras linhas das features (Grupo_pre ainda categórica; pode conter NaN,
# que serão imputados dentro da pipeline)
X.head()

,Idade,Sexo,Peso,Altura_cm,IMC,Grupo_pre,Fle_0,EVA_0,PM6_0,WD_0,WR_0,WAtotal_0,WT_0
0,67,0,80.0,158.0,32.046146,5,86.0,4.0,324.0,15,6.0,54.0,75
1,76,1,60.0,155.0,24.973985,5,90.0,4.0,357.0,14,7.0,39.0,60
2,72,1,93.0,182.0,28.076319,1,45.0,10.0,289.0,15,4.0,53.0,72
3,67,1,71.0,163.0,26.722873,6,127.0,2.0,390.0,1,1.0,16.0,18
4,66,0,49.0,160.0,19.140625,6,120.0,0.0,285.0,0,0.0,8.0,8


## 1) Preparação dos dados

As transformações determinísticas já foram aplicadas em `preprocessing.carregar_dados()`. Aqui apenas fazemos o **split estratificado único** (estratificado em `mudanca_CPAK`). A imputação da mediana e o one-hot encoding de `Grupo_pre` ficam dentro da pipeline (`preprocessing.construir_pipeline`), sendo ajustados **apenas com o treino** — o conjunto de teste é usado uma só vez, no fim.

In [3]:
# Valores nulos em X: existem por serem imputados dentro da pipeline (na mediana
# do treino de cada fold), e não à cabeça sobre todos os dados (evita leakage).
X.isnull().sum()

Idade        0
Sexo         0
Peso         2
Altura_cm    2
IMC          2
Grupo_pre    0
Fle_0        3
EVA_0        6
PM6_0        3
WD_0         0
WR_0         2
WAtotal_0    1
WT_0         0
dtype: int64

In [4]:
# Verificação de valores nulos em y
y.isnull().sum()

np.int64(0)

In [5]:
# Split estratificado único dos dados em treino e teste (feedback #1)
X_train, X_test, y_train, y_test = prep.dividir_treino_teste(X, y)

In [6]:
# Verificação da distribuição da target no conjunto de treino
print("Distribuição da Target (CPAK) - Treino")
print("N. observações Treino", len(X_train))
print(y_train.value_counts(normalize=True))

Distribuição da Target (CPAK) - Treino
N. observações Treino 183
0    0.901639
1    0.098361
Name: proportion, dtype: float64


In [7]:
# Verificação da distribuição da target no conjunto de teste

print("Distribuição da Target (CPAK) - Teste")
print("N. observações Teste", len(X_test))
print(y_test.value_counts(normalize=True))

Distribuição da Target (CPAK) - Teste
N. observações Teste 79
0    0.898734
1    0.101266
Name: proportion, dtype: float64


# 2) Obtenção de Baseline: Regressão logistica, Árvore de decisão e Random forest


## 2.1) Comparação dos três baselines com a mesma pipeline

Para evitar código duplicado (feedback #6), os três modelos passam pela **mesma** pipeline de pré-processamento (`preprocessing.construir_pipeline`), diferindo apenas no estimador. Cada um é:
1. treinado no conjunto de treino;
2. avaliado uma vez no teste — matriz de confusão + relatório + **AUC a partir de probabilidades** (feedback #2);
3. validado por **validação cruzada estratificada só no treino** (feedback #1).

In [8]:
# Três modelos baseline, todos com a mesma pipeline de pré-processamento.
# (LR é escalada porque beneficia de features na mesma escala.)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

modelos_baseline = {
    "Regressão Logística": (LogisticRegression(max_iter=1000, random_state=42), True),
    "Árvore de Decisão": (DecisionTreeClassifier(random_state=42), False),
    "Random Forest": (RandomForestClassifier(random_state=42), False),
}

resultados_baseline = {}
for nome, (modelo, escalar) in modelos_baseline.items():
    print("\n" + "=" * 60)
    print(nome)
    print("=" * 60)

    pipe = prep.construir_pipeline(modelo, X, escalar=escalar)
    pipe.fit(X_train, y_train)

    print("\n--- Avaliação no TESTE (usado uma só vez) ---")
    res = prep.avaliar_teste(pipe, X_test, y_test, titulo=nome)

    print("\n--- Validação cruzada estratificada (apenas no treino) ---")
    prep.validacao_cruzada_treino(pipe, X_train, y_train)

    resultados_baseline[nome] = res["auc"]


Regressão Logística

--- Avaliação no TESTE (usado uma só vez) ---
              precision    recall  f1-score   support

           0     0.9167    0.9296    0.9231        71
           1     0.2857    0.2500    0.2667         8

    accuracy                         0.8608        79
   macro avg     0.6012    0.5898    0.5949        79
weighted avg     0.8528    0.8608    0.8566        79

AUC (probabilidades): 0.7165



--- Validação cruzada estratificada (apenas no treino) ---


C:\Users\edu23\Desktop\Projetos_local\Machine_Learning\preprocessing.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9091    0.9697    0.9384       165
           1     0.2857    0.1111    0.1600        18

    accuracy                         0.8852       183
   macro avg     0.5974    0.5404    0.5492       183
weighted avg     0.8478    0.8852    0.8619       183

AUC CV (probabilidades): 0.8226

Árvore de Decisão

--- Avaliação no TESTE (usado uma só vez) ---


              precision    recall  f1-score   support

           0     0.9118    0.8732    0.8921        71
           1     0.1818    0.2500    0.2105         8

    accuracy                         0.8101        79
   macro avg     0.5468    0.5616    0.5513        79
weighted avg     0.8378    0.8101    0.8231        79

AUC (probabilidades): 0.5616

--- Validação cruzada estratificada (apenas no treino) ---


...

              precision    recall  f1-score   support

           0     0.9103    1.0000    0.9530        71
           1     1.0000    0.1250    0.2222         8

    accuracy                         0.9114        79
   macro avg     0.9551    0.5625    0.5876        79
weighted avg     0.9193    0.9114    0.8790        79

AUC (probabilidades): 0.7368

--- Validação cruzada estratificada (apenas no treino) ---


C:\Users\edu23\Desktop\Projetos_local\Machine_Learning\preprocessing.py:167: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


C:\Users\edu23\miniconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


              precision    recall  f1-score   support

           0     0.9116    1.0000    0.9538       165
           1     1.0000    0.1111    0.2000        18

    accuracy                         0.9126       183
   macro avg     0.9558    0.5556    0.5769       183
weighted avg     0.9203    0.9126    0.8796       183

AUC CV (probabilidades): 0.7813


In [9]:
# Tabela-resumo das AUC de teste (a partir de probabilidades) dos três baselines
tabela_baseline = (
    pd.Series(resultados_baseline, name="AUC_teste (probabilidades)")
    .sort_values(ascending=False)
    .to_frame()
)
tabela_baseline

,AUC_teste (probabilidades)
Random Forest,0.736796
Regressão Logística,0.716549
Árvore de Decisão,0.561620


# 3) Conclusão dos baselines e próximos passos

Com as métricas corrigidas (AUC calculada a partir de probabilidades e validação cruzada feita **apenas no treino**), a **Regressão Logística** e a **Random Forest** apresentam desempenho comparável e claramente superior ao da Árvore de Decisão em baseline.

O afinamento (tuning) de cada modelo é feito nos notebooks dedicados, que partilham o mesmo pré-processamento (`preprocessing.py`) e diferem apenas no modelo e nos seus hiperparâmetros:

- **`RL_limpo.ipynb`** — Regressão Logística (escolhida como modelo principal pela interpretabilidade e desempenho competitivo);
- **`AD_final.ipynb`** — Árvore de Decisão (com limitação de profundidade);
- **`RF.ipynb`** — Random Forest.

Em todos eles testa-se também o efeito do **SMOTE** sobre a classe minoritária.